In [ ]:
# =========================
# 0) Imports
# =========================
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
from google.colab import userdata
import time
import requests
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================
# 1) CONFIG: set these
# =========================
MODEL_ID = "gemma-3-1b"
SOURCE   = "22-gemmascope-2-res-262k"
BASE = "https://neuronpedia.org"
NEURONPEDIA_TOKEN = userdata.get("NEURONPEDIA_TOKEN")


K_PER_SEED = 800


N_EVAL = 600
BATCH_SIZE = 4


SCORE_MODE = "sum"
TOPK_FOR_MEAN = 5


POS_AUC_MIN = 0.62
NEG_AUC_MIN = 0.52
NEG_DELTA_MAX = -0.05

In [ ]:
# =========================
# 2) SEED TEXTS (diverse)
# =========================
POS_SEEDS = [
    "I loved this. It was amazing and made me happy.",
    "Fantastic and wonderful experience. Highly recommend.",
    "This is great—I'm impressed and satisfied.",
    "It was enjoyable, uplifting, and really good.",
    "I’m very pleased with the result.",
    "Excellent quality and super helpful.",
    "Brilliant work, absolutely loved it.",
    "This made my day. So happy.",
    "Perfect. Exactly what I wanted.",
    "I feel grateful and excited.",
    "A delightful experience with a positive outcome.",
    "I’m thrilled—everything worked beautifully.",
    "Really good service and friendly support.",
    "Superb. I’m impressed by how well this turned out.",
    "Warm, kind, and encouraging.",
]

NEG_SEEDS = [
    "I hated this. It was awful and made me angry.",
    "Terrible and disappointing experience. Do not recommend.",
    "This is bad—I'm upset and dissatisfied.",
    "It was frustrating, annoying, and really terrible.",
    "I regret buying this.",
    "Worst experience ever. Completely useless.",
    "This made me feel sad and disappointed.",
    "I’m disgusted and frustrated.",
    "Horrible quality. I want a refund.",
    "It’s unacceptable and painful to deal with.",
    "This was a waste of time and money.",
    "I’m furious—nothing worked and support was useless.",
    "Unpleasant, stressful, and disappointing.",
    "I feel miserable and let down.",
    "Awful. I strongly discourage anyone from trying this.",
]

In [ ]:
# =========================
# 3) Helpers: parse activeFeatures and score features
# =========================
def post_np(path, payload, retries=4, sleep=0.8):
    """
    Robust POST with retries + token.
    Header name can vary; based on your working calls, X-API-Key is most common.
    """
    url = f"{BASE}{path}"
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Colab)",
        "X-API-Key": NEURONPEDIA_TOKEN,
    }

    backoff = sleep
    last = None
    for i in range(retries):
        r = requests.post(url, headers=headers, json=payload, timeout=60)
        last = (r.status_code, r.text[:300])
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(backoff)
            backoff *= 2
            continue
        raise RuntimeError(f"HTTP {r.status_code} {path}: {r.text[:500]}")
    raise RuntimeError(f"Failed after retries. Last response: {last}")

def activation_source(model_id, source, custom_text):
    """
    POST /api/activation/source
    custom_text: str OR list[str] (up to 4)
    Returns JSON with key: 'results'
    """
    payload = {"modelId": model_id, "source": source, "customText": custom_text}
    return post_np("/api/activation/source", payload)

def top_features_from_result(result, k=200):
    """
    result: one dict with keys ["tokens", "activeFeatures", ...]
    activeFeatures maps feature_id -> list[[token_idx, value], ...]
    We rank features by total activation mass (sum across tokens).
    """
    af = result.get("activeFeatures", {})
    feat_scores = []
    for feat_str, spans in af.items():
        s = sum(v for (_, v) in spans)
        feat_scores.append((int(feat_str), float(s)))
    feat_scores.sort(key=lambda x: x[1], reverse=True)
    return [f for (f, _) in feat_scores[:k]]

def feature_scalar(active_features, feature_idx, mode="sum", topk=5):
    spans = active_features.get(str(feature_idx), [])
    if not spans:
        return 0.0
    vals = np.array([v for (_, v) in spans], dtype=np.float32)

    if mode == "sum":
        return float(vals.sum())
    if mode == "max":
        return float(vals.max())
    if mode == "mean_topk":
        k = min(topk, len(vals))
        return float(np.sort(vals)[-k:].mean())
    raise ValueError("Unknown mode")

def activation_source_batch(model_id, source, texts):
    """
    texts: list[str], length 1..4
    Returns list of result dicts (one per input text).
    """
    resp = activation_source(model_id, source, texts if len(texts) > 1 else texts[0])
    results = resp["results"]


    if len(texts) == 1:
        return results

    if len(results) != len(texts):

        out = []
        for t in texts:
            out.append(activation_source(model_id, source, t)["results"][0])
        return out

    return results

In [ ]:
# =========================
# 4) Candidate discovery: separate POS pool and NEG pool
# =========================
def discover_pool_from_seeds(model_id, source, seeds, k_per_seed=800):
    pool = set()
    for s in tqdm(seeds, desc="Discovering pool"):
        res = activation_source_batch(model_id, source, [s])[0]
        top_feats = top_features_from_result(res, k=k_per_seed)
        pool.update(top_feats)
    return pool

print("Discovering POS pool...")
pos_pool = discover_pool_from_seeds(MODEL_ID, SOURCE, POS_SEEDS, k_per_seed=K_PER_SEED)

print("Discovering NEG pool...")
neg_pool = discover_pool_from_seeds(MODEL_ID, SOURCE, NEG_SEEDS, k_per_seed=K_PER_SEED)

candidates = sorted(pos_pool | neg_pool)
print("POS pool:", len(pos_pool), "NEG pool:", len(neg_pool), "UNION:", len(candidates))

Discovering POS pool...


Discovering pool: 100%|██████████| 15/15 [00:15<00:00,  1.01s/it]


Discovering NEG pool...


Discovering pool: 100%|██████████| 15/15 [00:09<00:00,  1.51it/s]

POS pool: 2003 NEG pool: 2242 UNION: 3295


In [ ]:
# =========================
# 5) Load SST-2 for evaluation
# =========================
def load_sst2(n=600, split="validation", seed=0):
    ds = load_dataset("glue", "sst2", split=split)
    if n is not None and n < len(ds):
        ds = ds.shuffle(seed=seed).select(range(n))
    texts = [row["sentence"] for row in ds]
    labels = np.array([int(row["label"]) for row in ds], dtype=int)
    return texts, labels

texts, labels = load_sst2(n=N_EVAL, split="validation")

In [ ]:
# =========================
# 6) Evaluate candidates on SST-2 (batched)
# =========================
def evaluate_candidates(model_id, source, candidates, texts, labels,
                        score_mode="sum", topk=5, batch_size=4):
    scores = {c: [] for c in candidates}

    for i in tqdm(range(0, len(texts), batch_size), desc="SST-2 batching"):
        batch_texts = texts[i:i+batch_size]
        results = activation_source_batch(model_id, source, batch_texts)

        for res in results:
            af = res.get("activeFeatures", {})
            for c in candidates:
                scores[c].append(feature_scalar(af, c, mode=score_mode, topk=topk))

    rows = []
    for c in candidates:
        s = np.array(scores[c], dtype=np.float32)
        s = s[:len(labels)]

        try:
            auc = float(roc_auc_score(labels, s))
        except Exception:
            auc = float("nan")

        pos_mean = float(s[labels == 1].mean())
        neg_mean = float(s[labels == 0].mean())
        delta = pos_mean - neg_mean

        rows.append({
            "feature_index": c,
            "auc": auc,
            "pos_mean": pos_mean,
            "neg_mean": neg_mean,
            "delta_pos_minus_neg": delta,
            "direction": "positive" if delta > 0 else "negative",
        })

    df = pd.DataFrame(rows)


    delta_std = df["delta_pos_minus_neg"].std() + 1e-6
    df["rank_pos"] = df["auc"] + 0.25 * (df["delta_pos_minus_neg"] / delta_std)
    df["rank_neg"] = df["auc"] + 0.25 * ((-df["delta_pos_minus_neg"]) / delta_std)

    return df

df = evaluate_candidates(
    MODEL_ID, SOURCE, candidates, texts, labels,
    score_mode=SCORE_MODE, topk=TOPK_FOR_MEAN, batch_size=BATCH_SIZE
)

df.to_csv("/content/drive/MyDrive/ranked_sentiment_features.csv", index=False)
print("Saved ranked_sentiment_features.csv")

print("Max AUC:", df["auc"].max())

SST-2 batching: 100%|██████████| 150/150 [04:12<00:00,  1.68s/it]


Saved ranked_sentiment_features.csv
Max AUC: 0.7290128535847713


In [ ]:
# =========================
# 6.5) Filter overly-active / saturated features
# =========================
df = df.copy()

df["overall_mean"] = (df["pos_mean"] + df["neg_mean"]) / 2.0
high_act_cut = df["overall_mean"].quantile(0.98)


df = df[df["overall_mean"] <= high_act_cut].copy()

print("High-activation cutoff:", high_act_cut)
print("Remaining features after high-activation filter:", len(df))

High-activation cutoff: 824.3276562499984
Remaining features after high-activation filter: 3229


In [ ]:
# =========================
# 7) Select POS and NEG lists
# =========================
pos_cands = df[(df["auc"] >= POS_AUC_MIN) & (df["delta_pos_minus_neg"] > 0)].sort_values("rank_pos", ascending=False).head(3)
neg_cands = (df[df["delta_pos_minus_neg"] < -0.05].sort_values("rank_neg", ascending=False).head(3))

print("\nTop POS features:")
print(pos_cands.head(15)[["feature_index","auc","delta_pos_minus_neg","pos_mean","neg_mean"]].to_string(index=False))

print("\nTop NEG features:")
print(neg_cands.head(15)[["feature_index","auc","delta_pos_minus_neg","pos_mean","neg_mean"]].to_string(index=False))

pos_cands.head(50).to_csv("/content/drive/MyDrive/cs175/top_positive_features.csv", index=False)
neg_cands.head(50).to_csv("/content/drive/MyDrive/cs175/top_negative_features.csv", index=False)
print("\nSaved top_positive_features.csv and top_negative_features.csv")




Top POS features:
 feature_index      auc  delta_pos_minus_neg   pos_mean   neg_mean
          5210 0.688973           410.596146 555.870117 145.273972
          2219 0.631321           321.624619 445.850647 124.226028
          3770 0.622248           304.438873 621.740234 317.301361

Top NEG features:
 feature_index      auc  delta_pos_minus_neg   pos_mean   neg_mean
           176 0.158680          -913.122688  68.363640 981.486328
          4606 0.431779          -528.233765 405.766235 934.000000
          4773 0.373666          -500.701630 366.805206 867.506836

Saved top_positive_features.csv and top_negative_features.csv


In [ ]:
# =========================
# 8) Sanity check: show which tokens drive a feature
# =========================
def top_tokens_for_feature(model_id, source, feature_idx, text, top_n=8):
    res = activation_source_batch(model_id, source, [text])[0]
    toks = res["tokens"]
    spans = res["activeFeatures"].get(str(feature_idx), [])
    spans = sorted(spans, key=lambda x: x[1], reverse=True)[:top_n]
    return [(i, toks[i], float(v)) for i, v in spans]

def sanity_suite(feat):
    tests = {
        "pos_review": "Amazing! Loved it. 5 stars.",
        "neg_review": "Terrible. Hated it. 1 star.",
        "pos_nonreview": "I’m genuinely happy and satisfied today.",
        "neg_nonreview": "I feel awful, angry, and disappointed today.",
        "neutral": "The document describes the process and results."
    }
    print("\n==== Feature", feat, "====")
    for name, txt in tests.items():
        print(name, "=>", top_tokens_for_feature(MODEL_ID, SOURCE, feat, txt, top_n=6))

# Run sanity check on top 3 from each direction (if available)
print("\nSanity check (top POS):")
for f in pos_cands.head(3)["feature_index"].astype(int).tolist():
    sanity_suite(f)

print("\nSanity check (top NEG):")
for f in neg_cands.head(3)["feature_index"].astype(int).tolist():
    sanity_suite(f)


Sanity check (top POS):

==== Feature 5210 ====
pos_review => [(2, '!', 1632.0), (9, '.', 1072.0), (5, '.', 964.0), (6, ' ', 788.0)]
neg_review => []
pos_nonreview => [(9, '.', 268.0)]
neg_nonreview => [(4, ',', 260.0)]
neutral => []

==== Feature 2219 ====
pos_review => []
neg_review => []
pos_nonreview => []
neg_nonreview => []
neutral => []

==== Feature 3770 ====
pos_review => []
neg_review => []
pos_nonreview => []
neg_nonreview => []
neutral => []

Sanity check (top NEG):

==== Feature 176 ====
pos_review => []
neg_review => [(11, '.', 2512.0), (3, '.', 2368.0), (7, '.', 2240.0), (8, ' ', 1504.0), (2, 'rible', 1384.0), (6, ' it', 1304.0)]
pos_nonreview => []
neg_nonreview => [(4, ',', 760.0), (10, '.', 640.0), (5, ' angry', 592.0), (8, ' disappointed', 584.0), (3, ' awful', 544.0), (9, ' today', 528.0)]
neutral => []

==== Feature 4606 ====
pos_review => []
neg_review => []
pos_nonreview => []
neg_nonreview => [(3, ' awful', 360.0), (5, ' angry', 328.0)]
neutral => []

==== Feat